# CMES-86051 — Выбор 100 Gold ID и ослепление для человеческой оценки (Раздел 9)

Выбирает 100 Gold ID (`seed=42`), формирует 200 выходов (100 Static + 100
Dynamic), присваивает слепые ID, готовит форму для трёх независимых
журналистов, а затем (после того как формы заполнены) собирает всё в
`human_ratings_raw_anonymized.csv`.

Запускать **после** того, как готовы `static_predictions_1000_with_judge.jsonl`
и `dynamic_predictions_1000_with_judge.jsonl` (не обязательно ждать
GPT-judge — для человеческой оценки нужен только `generated_table`, но
логичнее делать оба этапа на одних и тех же финальных предсказаниях).


In [ ]:
import csv
import hashlib
import json
import random
from datetime import datetime, timezone
from pathlib import Path

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8-sig") as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

CONFIG = {
    "BENCHMARK": "CMES_evaluation/benchmark_gold_1000.jsonl",
    "STATIC_PREDICTIONS": "results_1000/static_predictions_1000.jsonl",
    "DYNAMIC_PREDICTIONS": "results_1000/dynamic_predictions_1000.jsonl",
    "SEED": 42,
    "N_IDS": 100,
    "OUTPUT_DIR": "human_eval",
    "N_RATERS": 3,
    "RATER_IDS": ["R1", "R2", "R3"],
}
Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)


In [ ]:
# ===================== ВЫБОР 100 GOLD ID (seed=42) =====================

benchmark = load_jsonl(CONFIG["BENCHMARK"])
static_preds = {str(r["id"]): r for r in load_jsonl(CONFIG["STATIC_PREDICTIONS"])}
dynamic_preds = {str(r["id"]): r for r in load_jsonl(CONFIG["DYNAMIC_PREDICTIONS"])}

all_ids = sorted(str(r["id"]) for r in benchmark)
assert len(all_ids) == 1000

rng = random.Random(CONFIG["SEED"])
selected_ids = sorted(rng.sample(all_ids, CONFIG["N_IDS"]))
print(f"Выбрано {len(selected_ids)} Gold ID, первые 10: {selected_ids[:10]}")


In [ ]:
# ===================== ФОРМИРОВАНИЕ 200 ВЫХОДОВ И ОСЛЕПЛЕНИЕ =====================

def generated_table_of(row):
    for key in ("generated_table", "generated", "table"):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value
    raise ValueError(f"id {row.get('id')!r}: пустой сгенерированный выход")

outputs = []
for record_id in selected_ids:
    for regime, source in (("static", static_preds), ("dynamic", dynamic_preds)):
        outputs.append({
            "gold_id": record_id,
            "hidden_regime": regime,
            "generated_table": generated_table_of(source[record_id]),
        })

assert len(outputs) == 200

# Перемешать порядок предъявления и присвоить слепые ID -- не по id/regime,
# чтобы не было систематического порядка (напр. static/dynamic подряд).
rng_shuffle = random.Random(CONFIG["SEED"])
rng_shuffle.shuffle(outputs)

for position, row in enumerate(outputs, 1):
    row["output_blind_id"] = f"B{position:04d}"
    row["presentation_order"] = position

print(outputs[0])


In [ ]:
# ===================== СОХРАНИТЬ MANIFEST (конфиденциально) =====================
# Этот файл содержит связку blind_id -> (gold_id, hidden_regime).
# НЕ показывать журналистам. Используется только на этапе объединения
# результатов после того, как все три оценки собраны.

manifest = {
    "created_utc": utc_now(),
    "seed": CONFIG["SEED"],
    "n_selected_ids": CONFIG["N_IDS"],
    "n_outputs": len(outputs),
    "n_raters": CONFIG["N_RATERS"],
    "rater_ids": CONFIG["RATER_IDS"],
    "selected_gold_ids": selected_ids,
    "blind_id_mapping": [
        {
            "output_blind_id": row["output_blind_id"],
            "gold_id": row["gold_id"],
            "hidden_regime": row["hidden_regime"],
            "presentation_order": row["presentation_order"],
        }
        for row in outputs
    ],
}
manifest_path = str(Path(CONFIG["OUTPUT_DIR"]) / "human_subset_manifest.json")
Path(manifest_path).write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(manifest_path)


In [ ]:
# ===================== ФОРМА ДЛЯ ЖУРНАЛИСТОВ (по одной на каждого) =====================
# output_blind_id + текст источника + сгенерированная таблица -- БЕЗ regime.
# Три идентичные формы (по порядку presentation_order), с пустыми колонками
# criterion_1..3 для заполнения. Дать одинаковую письменную инструкцию всем
# троим и убедиться, что они не видят оценки друг друга.

id_to_source_text = {str(r["id"]): r["text"] for r in benchmark}

fieldnames = [
    "output_blind_id", "presentation_order", "source_text", "generated_table",
    "rater_id", "criterion_1", "criterion_2", "criterion_3",
    "timestamp_utc", "comments_optional",
]

for rater_id in CONFIG["RATER_IDS"]:
    form_path = Path(CONFIG["OUTPUT_DIR"]) / f"rating_form_{rater_id}.csv"
    with open(form_path, "w", encoding="utf-8-sig", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for row in sorted(outputs, key=lambda r: r["presentation_order"]):
            writer.writerow({
                "output_blind_id": row["output_blind_id"],
                "presentation_order": row["presentation_order"],
                "source_text": id_to_source_text[row["gold_id"]],
                "generated_table": row["generated_table"],
                "rater_id": rater_id,
                "criterion_1": "",
                "criterion_2": "",
                "criterion_3": "",
                "timestamp_utc": "",
                "comments_optional": "",
            })
    print(form_path)


## Дальше — офлайн-этап

Передайте `rating_form_R1.csv`, `rating_form_R2.csv`, `rating_form_R3.csv`
(и общую письменную инструкцию с описанием 3 критериев 1–5) трём независимым
журналистам. Они заполняют `criterion_1..3` и `timestamp_utc` для каждой из
200 строк, не видя друг друга и не видя `human_subset_manifest.json`.

Когда все три формы заполнены и возвращены — положите их в
`human_eval/filled/rating_form_R1.csv` и т.д., и выполните ячейку ниже.


In [ ]:
# ===================== СБОРКА human_ratings_raw_anonymized.csv =====================

blind_to_meta = {
    row["output_blind_id"]: row for row in manifest["blind_id_mapping"]
}

filled_dir = Path(CONFIG["OUTPUT_DIR"]) / "filled"
combined_rows = []
for rater_id in CONFIG["RATER_IDS"]:
    filled_path = filled_dir / f"rating_form_{rater_id}.csv"
    with open(filled_path, encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            blind_id = row["output_blind_id"]
            meta = blind_to_meta[blind_id]
            for key in ("criterion_1", "criterion_2", "criterion_3"):
                val = int(row[key])
                assert val in (1, 2, 3, 4, 5), f"{blind_id}/{rater_id}/{key}: вне диапазона 1-5: {val}"
            combined_rows.append({
                "output_blind_id": blind_id,
                "gold_id": meta["gold_id"],
                "hidden_regime": meta["hidden_regime"],
                "rater_id": row["rater_id"],
                "criterion_1": row["criterion_1"],
                "criterion_2": row["criterion_2"],
                "criterion_3": row["criterion_3"],
                "timestamp_utc": row["timestamp_utc"],
                "comments_optional": row.get("comments_optional", ""),
            })

assert len(combined_rows) == 200 * CONFIG["N_RATERS"], (
    f"Ожидалось {200 * CONFIG['N_RATERS']} строк (200 outputs x {CONFIG['N_RATERS']} raters), "
    f"получено {len(combined_rows)}"
)

out_path = Path(CONFIG["OUTPUT_DIR"]) / "human_ratings_raw_anonymized.csv"
with open(out_path, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=[
        "output_blind_id", "gold_id", "hidden_regime", "rater_id",
        "criterion_1", "criterion_2", "criterion_3",
        "timestamp_utc", "comments_optional",
    ])
    writer.writeheader()
    writer.writerows(combined_rows)

print(out_path, "-", len(combined_rows), "строк")
